
# 02 - Data Preprocessing

## Project: Predictive Maintenance AI

### Objective
Prepare the AI4I 2020 dataset for Machine Learning.

### Tasks
1. Load the dataset
2. Inspect dataset structure
3. Remove irrelevant columns
4. Separate features and target
5. Encode categorical variables
6. Split training and testing data
7. Build preprocessing pipeline


In [1]:

import pandas as pd
import numpy as np

from pathlib import Path

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:

# Load original dataset

DATA_PATH = Path("../data/raw/ai4i2020.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")

print("Dataset Shape:", df.shape)

df.head()


Dataset loaded successfully!
Dataset Shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [3]:
# Dataset information

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9), str(2)
me

In [4]:
# Check missing values

print("\nMissing Values:")
print(df.isnull().sum().sum())


Missing Values:
0


In [5]:
# Check duplicate records

print("\nDuplicate Records:")
print(df.duplicated().sum())


Duplicate Records:
0


In [6]:

# Columns that should not be used as model inputs

columns_to_drop = [
    "UDI",
    "Product ID",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF"
]

# Create a new dataframe
df_model = df.drop(columns=columns_to_drop)

print("Original shape:", df.shape)
print("New shape:", df_model.shape)

df_model.head()


Original shape: (10000, 14)
New shape: (10000, 7)


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0


In [7]:

# Independent variables (features)
X = df_model.drop(columns=["Machine failure"])

# Dependent variable (target)
y = df_model["Machine failure"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())


Features shape: (10000, 6)
Target shape: (10000,)

Feature columns:
['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

Target distribution:
Machine failure
0    9661
1     339
Name: count, dtype: int64


In [10]:

from sklearn.model_selection import train_test_split

# Split dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Check dataset dimensions

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("\nTraining Target:", y_train.shape)
print("Testing Target:", y_test.shape)


Training Features: (8000, 6)
Testing Features: (2000, 6)

Training Target: (8000,)
Testing Target: (2000,)


In [11]:

print("Training Class Distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting Class Distribution:")
print(y_test.value_counts(normalize=True) * 100)


Training Class Distribution:
Machine failure
0    96.6125
1     3.3875
Name: proportion, dtype: float64

Testing Class Distribution:
Machine failure
0    96.6
1     3.4
Name: proportion, dtype: float64


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Separate numerical and categorical columns
numeric_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

categorical_features = ["Type"]

# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    )
}

# Combine preprocessing and each ML algorithm
model_pipelines = {}

for name, model in models.items():
    model_pipelines[name] = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

print("All three ML model pipelines created successfully!")

All three ML model pipelines created successfully!


In [14]:
training_results = {}

for name, pipeline in model_pipelines.items():

    print(f"Training {name}...")

    pipeline.fit(X_train, y_train)

    training_results[name] = pipeline

    print(f"{name} trained successfully!\n")

print("All models trained successfully!")

Training Logistic Regression...
Logistic Regression trained successfully!

Training Decision Tree...
Decision Tree trained successfully!

Training Random Forest...
Random Forest trained successfully!

All models trained successfully!


In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

evaluation_results = []

for name, pipeline in training_results.items():

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    evaluation_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "PR-AUC": average_precision_score(y_test, y_prob)
    })

results_df = pd.DataFrame(evaluation_results)

results_df = results_df.sort_values(
    by="PR-AUC",
    ascending=False
)

results_df.round(4)

,Model,Accuracy,Precision,Recall,F1 Score,PR-AUC
2,Random Forest,0.9800,0.9375,0.4412,0.6000,0.7802
1,Decision Tree,0.9790,0.7321,0.6029,0.6613,0.4549
0,Logistic Regression,0.8245,0.1418,0.8235,0.2419,0.3818


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.base import clone
from sklearn.metrics import precision_recall_curve
import numpy as np

# Create a validation set from training data
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

# Create a fresh Random Forest pipeline
rf_model = clone(model_pipelines["Random Forest"])

# Train on the smaller training portion
rf_model.fit(X_fit, y_fit)

# Get failure probabilities on validation data
val_probabilities = rf_model.predict_proba(X_val)[:, 1]

# Calculate precision and recall at different thresholds
precision, recall, thresholds = precision_recall_curve(
    y_val,
    val_probabilities
)

# Select the threshold with the highest precision
# while maintaining at least 80% recall
target_recall = 0.80

valid_indices = np.where(recall[:-1] >= target_recall)[0]

if len(valid_indices) > 0:
    best_index = valid_indices[
        np.argmax(precision[:-1][valid_indices])
    ]

    best_threshold = thresholds[best_index]

    print("Selected Threshold:", round(best_threshold, 4))
    print("Validation Precision:", round(precision[best_index], 4))
    print("Validation Recall:", round(recall[best_index], 4))

else:
    print("No threshold achieved the target recall.")

Selected Threshold: 0.1167
Validation Precision: 0.3793
Validation Recall: 0.8148


In [17]:
from sklearn.base import clone
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

# Create a fresh Random Forest pipeline
final_model = clone(model_pipelines["Random Forest"])

# Train using all training data
final_model.fit(X_train, y_train)

# Predict failure probabilities on test data
test_probabilities = final_model.predict_proba(X_test)[:, 1]

# Apply the threshold selected using validation data
final_predictions = (
    test_probabilities >= best_threshold
).astype(int)

# Evaluate final model
print("FINAL MODEL PERFORMANCE")
print("-----------------------")

print("Accuracy:", round(
    accuracy_score(y_test, final_predictions), 4
))

print("Precision:", round(
    precision_score(y_test, final_predictions, zero_division=0), 4
))

print("Recall:", round(
    recall_score(y_test, final_predictions, zero_division=0), 4
))

print("F1 Score:", round(
    f1_score(y_test, final_predictions, zero_division=0), 4
))

print("PR-AUC:", round(
    average_precision_score(y_test, test_probabilities), 4
))

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_test, final_predictions))

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_test,
    final_predictions,
    target_names=["Normal", "Failure"],
    zero_division=0
))

FINAL MODEL PERFORMANCE
-----------------------
Accuracy: 0.9565
Precision: 0.4296
Recall: 0.8529
F1 Score: 0.5714
PR-AUC: 0.7802

CONFUSION MATRIX
[[1855   77]
 [  10   58]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

      Normal       0.99      0.96      0.98      1932
     Failure       0.43      0.85      0.57        68

    accuracy                           0.96      2000
   macro avg       0.71      0.91      0.77      2000
weighted avg       0.98      0.96      0.96      2000



In [18]:
import joblib
from pathlib import Path

# Create models folder if it doesn't exist
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save model and threshold together
model_bundle = {
    "model": final_model,
    "threshold": float(best_threshold),
    "model_name": "Random Forest",
    "features": X_train.columns.tolist(),
    "target": "Machine failure"
}

model_path = MODEL_DIR / "predictive_maintenance_model.joblib"

joblib.dump(model_bundle, model_path)

print("Model saved successfully!")
print("Saved location:", model_path)

Model saved successfully!
Saved location: ..\models\predictive_maintenance_model.joblib


In [19]:
loaded_bundle = joblib.load(model_path)

loaded_model = loaded_bundle["model"]
loaded_threshold = loaded_bundle["threshold"]

print("Model loaded successfully!")
print("Model:", loaded_bundle["model_name"])
print("Threshold:", loaded_threshold)
print("Features:", loaded_bundle["features"])

Model loaded successfully!
Model: Random Forest
Threshold: 0.11666666666666667
Features: ['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
